In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [2]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

2025/05/29 06:35:45 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location=('/workspaces/MLOps-ZoomCamp/02-Experiment tracking and model '
 'management/mlruns/1'), creation_time=1748500545859, experiment_id='1', last_update_time=1748500545859, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [3]:
def read_dataframe(filename):
    if filename.endswith(".csv"):
        df = pd.read_csv(filename)

        df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
        df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

    elif filename.endswith(".parquet"):
        df = pd.read_parquet(filename)

    df["duration"] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    return df

In [4]:
df_train = read_dataframe("../data/yellow_tripdata_2023-01.parquet")
df_val = read_dataframe("../data/yellow_tripdata_2023-02.parquet")

In [5]:
categorical = ["PULocationID", "DOLocationID"]

# Turn dataframes into list of dictionaries
train_dicts = df_train[categorical].to_dict(orient="records")
val_dicts = df_val[categorical].to_dict(orient="records")

# Set GT Values
y_train = df_train["duration"].values
y_val = df_val["duration"].values

: 

In [ ]:

# Fit dictionary vectorizer on Training data
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

# Get dimensionality
print(f"Q4 Answer - Dimensionality of feature matrix: {X_train.shape[1]}")

In [ ]:

# Train linear regression model 
lr = LinearRegression()
lr.fit(X_train, y_train)

# Make predictions/infer on training data
y_pred_train = lr.predict(X_train)

# Calculate RMSE on training data
rmse_train = root_mean_squared_error(y_train, y_pred_train)
print(f"Q5 Answer - RMSE on train: {rmse_train:.2f}")

In [ ]:

# Apply learned dictionary vectorizer on validation data 
X_val = dv.transform(val_dicts)

# Make predictions on validation data
y_pred_val = lr.predict(X_val)

# Calculate RMSE on validation
rmse_val = root_mean_squared_error(y_val, y_pred_val)
print(f"Q6 Answer - RMSE on validation: {rmse_val:.2f}")